In [ ]:
from wave_client import WaveClient
import pandas as pd
from utils import (
    load_environment_variables,
)


In [ ]:
# Load environment variables and connect to WAVE backend
RESEARCHER_API_KEY, EXPERIMENTEE_API_KEY, WAVE_BACKEND_URL = load_environment_variables()

print(f"🔍 Connecting to WAVE backend...")
print(f"📡 Backend URL: {WAVE_BACKEND_URL}")
print(f"🔑 Using Researcher API key ending in: ...{RESEARCHER_API_KEY[-4:]}")

client = WaveClient(api_key=RESEARCHER_API_KEY, base_url=WAVE_BACKEND_URL)

In [ ]:
## use this (uncomment it) to see what commands exist 
## (allow the scrollable option at the bottom to see all commands)

# help(client.experiments)
# help(client.experiment_data)

In [ ]:
# Get all experiments to see which one to analyze
experiments = await client.experiments.list(skip=0, limit=200)
print(f"Found {len(experiments)} experiments\n")

# Sort by creation date (most recent first) and show up to 5
sorted_experiments = sorted(experiments, key=lambda x: x.get('created_at', ''), reverse=True)[:5]
print(f"Showing {len(sorted_experiments)} most recent experiments (of {len(experiments)} total)\n")

# Display experiment info
for i, exp in enumerate(sorted_experiments, 1):
    print(f"{i}. Experiment UUID: {exp['uuid']}")
    print(f"   Description: {exp.get('description', 'N/A')}")
    print(f"   Type ID: {exp.get('experiment_type_id', 'N/A')}")
    print(f"   Created: {exp.get('created_at', 'N/A')[:10]}")
    print(f"   Tags: {', '.join(exp.get('tags', []))}")
    print("-" * 70)


In [ ]:
# Put the necessary info into the variables below 
selected_experiment_uuid = "bb4d13ff-59a7-4804-8bb1-a2d6f26b5e36" 

# Get all data from that experiment
print(f"\n📊 Fetching data for experiment {selected_experiment_uuid}...")
experiment_data_df = await client.experiment_data.get_all_data(
    experiment_id=selected_experiment_uuid
)

print(f"Total records: {len(experiment_data_df)}")
# print(experiment_data_df)


In [ ]:
# Filter and save into .csv file
datafolder = "data/" #this is what Kim uses
# datafolder = "/Users/evaliss/Desktop/WAVElab_code/epCD/data/pilot 1/" #this is what Eva uses

# experiment_data_df.to_csv(f"data/temp.csv", index=False) #for troubleshooting

# full_df = (experiment_data_df.sort_values(by="timestamp",ascending=True))
full_df = (experiment_data_df.query("not participant_id.str.contains('PROLIFIC_PID') and not participant_id.str.contains('no_query')")
           .sort_values(by="timestamp",ascending=True))
answer_df = full_df.query("trial_category == 'answerexpt'")
debrief_df = full_df.query("trial_category == 'debriefexpt'")

full_df.to_csv(f"{datafolder}dataframe_full.csv", index = False)
answer_df.to_csv(f"{datafolder}dataframe_answer.csv", index = False)
debrief_df.to_csv(f"{datafolder}dataframe_debrief.csv", index = False)



In [ ]:
## get debrief attention
debrief_attention = (
    debrief_df["response"]
    .apply(lambda r: pd.json_normalize(eval(r)).get("attention", pd.Series([None]))[0] if pd.notna(r) else None)
)
debrief_df = debrief_df.copy()
debrief_df["participant_attention"] = debrief_attention

participant_attention_map = debrief_df.drop_duplicates("participant_id").set_index("participant_id")["participant_attention"]
answer_df["participant_attention"] = answer_df["participant_id"].map(participant_attention_map)
answer_df["participant_attention"] = pd.to_numeric(answer_df["participant_attention"], errors='coerce')
debrief_df["participant_attention"] = pd.to_numeric(debrief_df["participant_attention"], errors='coerce')
full_df["participant_attention"] = full_df["participant_id"].map(participant_attention_map)

debrief_df.to_csv(f"{datafolder}dataframe_debrief.csv", index=False)
full_df.to_csv(f"{datafolder}dataframe_full.csv", index=False)
answer_df.to_csv(f"{datafolder}dataframe_answer.csv", index=False)

print("Attention Breakdown:")
print(debrief_df["participant_attention"].value_counts().sort_index(ascending=False).to_string())

In [ ]:
# For creating new columns based on the information in firstStim & secondStim

stimColumns = ["firstStim","secondStim"]

for column in stimColumns:
    selected_column = answer_df[column]
    position_column = []

    for item in selected_column:
        if "Center" in item: 
            position_column.append("Center")
        elif "Right" in item:
            position_column.append("Right")
        else:
            position_column.append("Error")
    
    answer_df[column+"_cupPos"] = position_column
    answer_df.to_csv("data/dataframe_answer.csv", index = False)


